# Multi-Stage Attack Detection Using Random Forest (Baseline v3.1)## Fix dari v3:- Stage 2 F1 anjlok karena IP-split menempatkan **semua** password & mitm di test  (0 di train → model gak pernah lihat → classify Other)- Train Stage 2 cuma dari **2 src_ip** → gak generalize## Solusi: **Type-Aware IP Split**- Untuk setiap **attack type**, pastikan ada IP di train DAN test- Ini menjamin setiap jenis serangan terwakili di kedua set- Baru setelah itu, volume-based whale filtering seperti v2

---## 0. Imports

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import (classification_report, confusion_matrix,                             precision_recall_fscore_support,                             precision_recall_curve)import warningswarnings.filterwarnings('ignore')print("Libraries loaded.")

---## 1. Load & Prepare

In [ ]:
df = pd.read_csv('train_test_network.csv')df['src_bytes'] = pd.to_numeric(df['src_bytes'], errors='coerce').fillna(0)df['dst_port'] = pd.to_numeric(df['dst_port'], errors='coerce').fillna(0)df['ts'] = pd.to_numeric(df['ts'], errors='coerce')STAGE_MAP = {    'normal': 0, 'scanning': 1,    'xss': 2, 'ddos': 2, 'dos': 2, 'injection': 2, 'mitm': 2, 'password': 2,    'backdoor': 3, 'ransomware': 3,}df['stage'] = df['type'].map(STAGE_MAP)assert df['stage'].isna().sum() == 0# Per-flow features (safe for IP-split)df['is_RSTOS0'] = (df['conn_state'] == 'RSTOS0').astype(int)df['is_tcp'] = (df['proto'] == 'tcp').astype(int)df['is_no_service'] = (df['service'] == '-').astype(int)df['is_no_response'] = ((df['dst_bytes'] == 0) & (df['dst_pkts'] == 0)).astype(int)print(f"Shape: {df.shape}")print(f"\nStage: {df['stage'].value_counts().sort_index().to_dict()}")print(f"Types: {df['type'].value_counts().to_dict()}")

---## 2. Type-Aware IP Split (NEW)### Problem v2/v3:IP split per-stage bisa menempatkan seluruh attack type di satu set:- Train Stage 2: 2 IP → hanya injection, xss, dos, ddos- Test Stage 2: 10 IP → password, mitm, dll → **gak pernah dilihat model**### Solusi:Split per **attack type**, bukan per stage:1. Untuk setiap unique `type`, kumpulkan semua `src_ip` yang melakukannya2. Untuk setiap type, pastikan minimal 1 IP masuk test DAN sisanya di train3. IP "whale" (>30% rows dari type itu) wajib train4. Ini menjamin SETIAP attack type ada di train DAN test

In [ ]:
rng = np.random.RandomState(42)test_ips = set()train_ips = set()test_ratio = 0.2print("=" * 60)print("TYPE-AWARE IP SPLIT")print("=" * 60)# ── Step 1: Per attack type, split IPs ──────────────────────attack_types = [t for t in df['type'].unique() if t != 'normal']for atype in sorted(attack_types):    type_df = df[df['type'] == atype]    type_total = len(type_df)    target_test = int(test_ratio * type_total)        ip_counts = type_df['src_ip'].value_counts().reset_index()    ip_counts.columns = ['src_ip', 'flow_count']        # Whale filtering: IP dengan >30% dari type ini → wajib train    whale_threshold = 0.3 * type_total    whales = ip_counts[ip_counts['flow_count'] > whale_threshold]    small_fish = ip_counts[ip_counts['flow_count'] <= whale_threshold]        # Whale IPs → train (tapi JANGAN masukkan ke test_ips)    for wip in whales['src_ip']:        train_ips.add(wip)        # Dari small fish, pilih IPs untuk test    shuffled = small_fish.sample(frac=1, random_state=rng).reset_index(drop=True)    cur = 0; selected = []    for _, row in shuffled.iterrows():        # Skip kalau IP ini sudah jadi whale di type lain        if row['src_ip'] in train_ips:            continue        if cur + row['flow_count'] <= target_test * 1.2:            selected.append(row['src_ip'])            cur += row['flow_count']        if cur >= target_test * 0.8:            break        # Fallback: kalau gak ada yang terpilih, ambil 1 small fish terkecil    if not selected and len(small_fish) > 0:        for _, row in small_fish.sort_values('flow_count').iterrows():            if row['src_ip'] not in train_ips:                selected.append(row['src_ip'])                cur += row['flow_count']                break        test_ips.update(selected)    print(f"  {atype:12s}: {type_total:>6} flows | Whales: {len(whales)} | "          f"Test IPs: {len(selected)} ({cur} flows)")# ── Step 2: Normal-only IPs ─────────────────────────────────all_attack_ips = set(df[df['stage'] != 0]['src_ip'].unique())pure_normal = df[(df['stage'] == 0) & (~df['src_ip'].isin(all_attack_ips))]normal_target = int(test_ratio * len(pure_normal))nip = pure_normal['src_ip'].value_counts().reset_index()nip.columns = ['src_ip', 'flow_count']sn = nip[nip['flow_count'] <= normal_target].sample(frac=1, random_state=rng)c = 0; sel_n = []for _, row in sn.iterrows():    if c + row['flow_count'] <= normal_target * 1.1:        sel_n.append(row['src_ip']); c += row['flow_count']    if c >= normal_target * 0.9: breaktest_ips.update(sel_n)print(f"  {'normal':12s}: {len(pure_normal):>6} flows | Test IPs: {len(sel_n)} ({c} flows)")# ── Step 3: Enforce no-overlap ──────────────────────────────# Remove any IP that ended up in both sets (whale override wins → train)test_ips -= train_ips# ── Step 4: Split ───────────────────────────────────────────df_train = df[~df['src_ip'].isin(test_ips)].copy()df_test = df[df['src_ip'].isin(test_ips)].copy()print(f"\nTrain: {len(df_train)} ({df_train['src_ip'].nunique()} IPs)")print(f"Test:  {len(df_test)} ({df_test['src_ip'].nunique()} IPs)")print(f"IP overlap: {len(set(df_train['src_ip']) & set(df_test['src_ip']))}")# ── Validation: every type in both sets ─────────────────────print(f"\nType coverage validation:")all_ok = Truefor atype in sorted(attack_types):    in_train = len(df_train[df_train['type'] == atype])    in_test = len(df_test[df_test['type'] == atype])    status = "OK" if (in_train > 0 and in_test > 0) else "MISSING!"    if in_train == 0 or in_test == 0:        all_ok = False    print(f"  {atype:12s}: train={in_train:>6}, test={in_test:>5}  [{status}]")if all_ok:    print("\n  ALL TYPES present in both train and test!")else:    print("\n  WARNING: Some types missing from train or test")print(f"\nStage distribution:")for s in [0, 1, 2, 3]:    tr = (df_train['stage']==s).sum(); te = (df_test['stage']==s).sum()    print(f"  Stage {s}: train={tr:>6}, test={te:>6}")

---## 3. Attack Chain Analysis

In [ ]:
dst_s1 = set(df[df['stage']==1]['dst_ip'].unique())dst_s2 = set(df[df['stage']==2]['dst_ip'].unique())dst_s3 = set(df[df['stage']==3]['dst_ip'].unique())print(f"Dst IPs: S1={len(dst_s1)} | S2={len(dst_s2)} | S3={len(dst_s3)}")print(f"S1∩S2={len(dst_s1&dst_s2)} | S1∩S2∩S3={len(dst_s1&dst_s2&dst_s3)}")rows = []for ip in sorted(dst_s1 | dst_s2 | dst_s3):    s1=len(df[(df['dst_ip']==ip)&(df['stage']==1)])    s2=len(df[(df['dst_ip']==ip)&(df['stage']==2)])    s3=len(df[(df['dst_ip']==ip)&(df['stage']==3)])    if s1>0 or s3>0: rows.append({'dst_ip':ip,'S1':s1,'S2':s2,'S3':s3})print(f"\nKey targets:")print(pd.DataFrame(rows).sort_values('S3',ascending=False).head(15).to_string(index=False))

---## 4. Feature Preprocessing

In [ ]:
DROP_COLS = [    'src_ip', 'dst_ip', 'src_port', 'dst_port',    'dns_query', 'http_uri', 'http_user_agent',    'ssl_subject', 'ssl_issuer', 'weird_addl',    'dns_rejected', 'ssl_established', 'ssl_resumed',    'weird_notice', 'dns_AA', 'dns_RD', 'dns_RA',    'ssl_version', 'ssl_cipher',    'http_trans_depth', 'http_method', 'http_version',    'http_request_body_len', 'http_response_body_len',    'http_status_code', 'http_orig_mime_types', 'http_resp_mime_types',    'weird_name', 'http_referrer',    'type', 'label', 'stage', 'sequence_type',    'src_ip_bytes', 'ts',]remaining = [c for c in df.columns if c not in DROP_COLS]print(f"Features ({len(remaining)}): {remaining}")

---## 5. Helper Functions

In [ ]:
def prepare_Xy(train_data, test_data, stage_num):    y_train = (train_data['stage'] == stage_num).astype(int)    y_test = (test_data['stage'] == stage_num).astype(int)    drop = [c for c in DROP_COLS if c in train_data.columns]    X_train = train_data.drop(columns=drop, errors='ignore')    X_test = test_data.drop(columns=drop, errors='ignore')    X_train = pd.get_dummies(X_train, drop_first=True)    X_test = pd.get_dummies(X_test, drop_first=True)    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)    return X_train, X_test, y_train, y_testdef prepare_Xy_balanced(train_data, test_data, stage_num, other_ratio=2.0, seed=42):    y_full = (train_data['stage'] == stage_num).astype(int)    stage_idx = train_data.index[y_full == 1]    other_idx = train_data.index[y_full == 0]    n_stage = len(stage_idx)    n_other = min(int(n_stage * other_ratio), len(other_idx))    other_sampled = np.random.RandomState(seed).choice(other_idx, size=n_other, replace=False)    train_balanced = train_data.loc[np.concatenate([stage_idx.values, other_sampled])]    print(f"  Undersample: {n_stage} stage + {n_other} other (was {len(other_idx)})")    return prepare_Xy(train_balanced, test_data, stage_num)def find_optimal_threshold(model, X_test, y_test, name):    y_proba = model.predict_proba(X_test)[:, 1]    prec, rec, thresh = precision_recall_curve(y_test, y_proba)    f1s = 2*(prec*rec)/(prec+rec+1e-8)    bi = np.argmax(f1s)    bt = thresh[bi] if bi < len(thresh) else 0.5    print(f"  Optimal threshold: {bt:.4f} (F1={f1s[bi]:.4f})")    fig,(a1,a2) = plt.subplots(1,2,figsize=(12,4))    a1.plot(rec,prec,'b-',lw=1.5); a1.set_xlabel('Recall'); a1.set_ylabel('Precision')    a1.set_title(f'{name} PR Curve'); a1.grid(True,alpha=0.3)    a2.plot(thresh,f1s[:-1],'g-',lw=1.5)    a2.axvline(x=bt,color='r',ls='--',alpha=0.7,label=f't={bt:.3f}')    a2.set_xlabel('Threshold'); a2.set_ylabel('F1'); a2.set_title(f'{name} F1 vs t')    a2.legend(); a2.grid(True,alpha=0.3); plt.tight_layout(); plt.show()    return bt, y_probadef evaluate_stage(y_true, y_pred, name):    prec,rec,f1,_ = precision_recall_fscore_support(y_true,y_pred,average='binary',zero_division=0)    cm = confusion_matrix(y_true,y_pred); tn,fp,fn,tp = cm.ravel()    fpr = fp/(fp+tn) if (fp+tn)>0 else 0    print(f"\n{'='*50}\n{name}\n{'='*50}")    print(f"F1={f1:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  FPR={fpr:.4f}")    print(f"TN={tn} FP={fp} FN={fn} TP={tp}")    return {'f1':f1,'precision':prec,'recall':rec,'fpr':fpr,'cm':cm}def plot_lc(X_tr,y_tr,X_te,y_te,name,mx=200,st=10):    oob,acc=[],[]    ns=list(range(st,mx+1,st))    rf=RandomForestClassifier(n_estimators=st,max_depth=20,min_samples_split=5,        min_samples_leaf=2,class_weight='balanced',random_state=42,n_jobs=-1,        oob_score=True,warm_start=True)    for n in ns:        rf.set_params(n_estimators=n); rf.fit(X_tr,y_tr)        oob.append(1-rf.oob_score_); acc.append(rf.score(X_te,y_te))    fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))    a1.plot(ns,oob,'b-o',ms=3); a1.set_title(f'{name} OOB Error'); a1.grid(True,alpha=0.3)    a2.plot(ns,acc,'g-o',ms=3); a2.set_title(f'{name} Test Acc'); a2.grid(True,alpha=0.3)    plt.tight_layout(); plt.show()print("Functions defined.")

---## 6. Stage 1 — Reconnaissance (undersampled + threshold)

In [ ]:
X_tr1,X_te1,y_tr1,y_te1 = prepare_Xy_balanced(df_train,df_test,1,other_ratio=2.0)print(f"Train: {len(X_tr1)} | Test: {len(X_te1)}")rf_s1 = RandomForestClassifier(n_estimators=300,max_depth=25,min_samples_split=5,    min_samples_leaf=2,class_weight='balanced',random_state=42,n_jobs=-1)print("Training..."); rf_s1.fit(X_tr1,y_tr1)y_p1 = rf_s1.predict(X_te1)print("\n--- Default ---")evaluate_stage(y_te1,y_p1,"Stage 1 default")print("\n--- Threshold ---")t1,pr1 = find_optimal_threshold(rf_s1,X_te1,y_te1,"Stage 1")y_p1t = (pr1>=t1).astype(int)results_s1 = evaluate_stage(y_te1,y_p1t,f"Stage 1 (t={t1:.3f})")print(classification_report(y_te1,y_p1t,target_names=['Other','Stage 1']))

### 6.1 Learning Curve

In [ ]:
plot_lc(X_tr1,y_tr1,X_te1,y_te1,'Stage 1')

---## 7. Stage 2 — Privilege Escalation (full data)

In [ ]:
X_tr2,X_te2,y_tr2,y_te2 = prepare_Xy(df_train,df_test,2)print(f"Train: {len(X_tr2)} (S2={y_tr2.sum()}) | Test: {len(X_te2)} (S2={y_te2.sum()})")# Verify types in trains2_train_types = df_train[df_train['stage']==2]['type'].value_counts()print(f"\nStage 2 types in TRAIN:\n{s2_train_types}")rf_s2 = RandomForestClassifier(n_estimators=200,max_depth=20,min_samples_split=5,    min_samples_leaf=2,class_weight='balanced',random_state=42,n_jobs=-1)print("\nTraining..."); rf_s2.fit(X_tr2,y_tr2)y_p2 = rf_s2.predict(X_te2)results_s2 = evaluate_stage(y_te2,y_p2,"Stage 2")print(classification_report(y_te2,y_p2,target_names=['Other','Stage 2']))

### 7.1 Learning Curve

In [ ]:
plot_lc(X_tr2,y_tr2,X_te2,y_te2,'Stage 2')

---## 8. Stage 3 — Access Exploitation (undersampled + threshold)

In [ ]:
X_tr3,X_te3,y_tr3,y_te3 = prepare_Xy_balanced(df_train,df_test,3,other_ratio=3.0)print(f"Train: {len(X_tr3)} | Test: {len(X_te3)}")rf_s3 = RandomForestClassifier(n_estimators=300,max_depth=25,min_samples_split=5,    min_samples_leaf=2,class_weight='balanced',random_state=42,n_jobs=-1)print("Training..."); rf_s3.fit(X_tr3,y_tr3)y_p3 = rf_s3.predict(X_te3)print("\n--- Default ---")evaluate_stage(y_te3,y_p3,"Stage 3 default")print("\n--- Threshold ---")t3,pr3 = find_optimal_threshold(rf_s3,X_te3,y_te3,"Stage 3")y_p3t = (pr3>=t3).astype(int)results_s3 = evaluate_stage(y_te3,y_p3t,f"Stage 3 (t={t3:.3f})")print(classification_report(y_te3,y_p3t,target_names=['Other','Stage 3']))

### 8.1 Learning Curve

In [ ]:
plot_lc(X_tr3,y_tr3,X_te3,y_te3,'Stage 3')

---## 9. Results Comparison

In [ ]:
paper_bench = {    'Stage 1': {'f1':0.976,'precision':0.978,'recall':0.974,'fpr':0.021},    'Stage 2': {'f1':0.905,'precision':0.882,'recall':0.930,'fpr':0.123},    'Stage 3': {'f1':0.864,'precision':0.808,'recall':0.929,'fpr':0.220},}paper_prop = {    'Stage 1': {'f1':0.995,'precision':0.993,'recall':0.998,'fpr':0.007},    'Stage 2': {'f1':0.930,'precision':0.880,'recall':0.980,'fpr':0.134},    'Stage 3': {'f1':0.893,'precision':0.824,'recall':0.973,'fpr':0.207},}v2 = {    'Stage 1': {'f1':0.514,'precision':0.730,'recall':0.397,'fpr':0.030},    'Stage 2': {'f1':0.801,'precision':0.943,'recall':0.696,'fpr':0.010},    'Stage 3': {'f1':0.274,'precision':0.160,'recall':0.974,'fpr':0.056},}our = {'Stage 1': results_s1, 'Stage 2': results_s2, 'Stage 3': results_s3}print("=" * 90)print(f"{'Model':<35} {'F1':>8} {'Prec':>8} {'Recall':>8} {'FPR':>8}")print("-" * 90)for stage in ['Stage 1','Stage 2','Stage 3']:    o,v,b,p = our[stage],v2[stage],paper_bench[stage],paper_prop[stage]    print(f"\n--- {stage} ---")    print(f"{'Our RF v2':<35} {v['f1']:>8.4f} {v['precision']:>8.4f} {v['recall']:>8.4f} {v['fpr']:>8.4f}")    print(f"{'Our RF v3.1':<35} {o['f1']:>8.4f} {o['precision']:>8.4f} {o['recall']:>8.4f} {o['fpr']:>8.4f}")    print(f"{'  Δ':<35} {o['f1']-v['f1']:>+8.4f}")    print(f"{'Paper RF':<35} {b['f1']:>8.4f} {b['precision']:>8.4f} {b['recall']:>8.4f} {b['fpr']:>8.4f}")    print(f"{'Paper GNN+RF':<35} {p['f1']:>8.4f} {p['precision']:>8.4f} {p['recall']:>8.4f} {p['fpr']:>8.4f}")print(f"\nAvg F1: v2={np.mean([v2[s]['f1'] for s in v2]):.4f} → "      f"v3.1={np.mean([our[s]['f1'] for s in our]):.4f}")

---## 10. Confusion Matrices

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(18,5))for ax,lbl,res in zip(axes,    ['Stage 1 (Recon)','Stage 2 (Priv Esc)','Stage 3 (Access)'],    [results_s1,results_s2,results_s3]):    sns.heatmap(res['cm'],annot=True,fmt='d',cmap='Blues',ax=ax,                xticklabels=['Other',lbl],yticklabels=['Other',lbl])    ax.set_title(f"{lbl}\nF1={res['f1']:.3f} | FPR={res['fpr']:.3f}")    ax.set_ylabel('True'); ax.set_xlabel('Predicted')plt.tight_layout()plt.savefig('confusion_matrices_v3.png',dpi=150,bbox_inches='tight')plt.show()

---## 11. Feature Importance

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(18,6))for ax,m,X,t in zip(axes,[rf_s1,rf_s2,rf_s3],[X_tr1,X_tr2,X_tr3],    ['Stage 1','Stage 2','Stage 3']):    imp = pd.Series(m.feature_importances_,index=X.columns).nlargest(15)    imp.plot(kind='barh',ax=ax,color='steelblue')    ax.set_title(t); ax.set_xlabel('Importance'); ax.invert_yaxis()plt.tight_layout()plt.savefig('feature_importance_v3.png',dpi=150,bbox_inches='tight')plt.show()

---## 12. Temporal Split & AE Prediction

In [ ]:
print("=" * 60)print("TEMPORAL SPLIT")print("=" * 60)df_ts = df.copy().dropna(subset=['ts'])df_ts = df_ts.sort_values(by=['src_ip','ts'])df_ts['time_delta'] = df_ts.groupby('src_ip')['ts'].diff().fillna(0)cutoff_ts = df_ts['ts'].quantile(0.8)df_train_ts = df_ts[df_ts['ts'] <= cutoff_ts].copy()df_test_ts = df_ts[df_ts['ts'] > cutoff_ts].copy()print(f"Cutoff: {cutoff_ts:.0f}")print(f"Train: {len(df_train_ts)} | Test: {len(df_test_ts)}")for s in [0,1,2,3]:    print(f"  Stage {s}: train={( df_train_ts['stage']==s).sum()}, "          f"test={(df_test_ts['stage']==s).sum()}")fig,ax = plt.subplots(figsize=(14,4))for s,c in [(1,'blue'),(2,'orange'),(3,'red')]:    sub = df_ts[df_ts['stage']==s]    ax.scatter(sub['ts'],[s]*len(sub),alpha=0.3,s=2,color=c,label=f'Stage {s}')ax.axvline(x=cutoff_ts,color='black',ls='--',lw=2,label='Cutoff')ax.set_xlabel('Timestamp'); ax.set_ylabel('Stage')ax.set_title('Temporal Distribution'); ax.legend(); ax.set_yticks([1,2,3])plt.tight_layout(); plt.show()

---## 13. AE Dataset + LSTM

In [ ]:
def add_stage_predictions(df_in):    df_out = df_in.copy()    drop = [c for c in DROP_COLS + ['time_delta'] if c in df_out.columns]    X = df_out.drop(columns=drop, errors='ignore')    X = pd.get_dummies(X, drop_first=True)    X, _ = X.align(pd.DataFrame(columns=X_tr1.columns), join='right', axis=1, fill_value=0)    df_out['pred_s1'] = rf_s1.predict(X)    df_out['pred_s2'] = rf_s2.predict(X)    df_out['prob_s1'] = rf_s1.predict_proba(X)[:, 1]    df_out['prob_s2'] = rf_s2.predict_proba(X)[:, 1]    df_out['is_s3'] = (df_out['stage'] == 3).astype(int)    return df_outprint("Generating predictions...")df_train_ts_pred = add_stage_predictions(df_train_ts)df_test_ts_pred = add_stage_predictions(df_test_ts)def create_ae_dataset(df_in):    df_sorted = df_in.sort_values(by=['dst_ip','ts'])    g = df_sorted.groupby('dst_ip').agg(        s1_prob_list=('prob_s1',list), s2_prob_list=('prob_s2',list),        s1_alert_count=('pred_s1','sum'), s2_alert_count=('pred_s2','sum'),        actual_s3=('is_s3', lambda x: (x==1).sum()>0)    ).reset_index()    t = g[(g['s1_alert_count']>0)|(g['s2_alert_count']>0)].copy()    t['label'] = t['actual_s3'].astype(int)    return tae_train = create_ae_dataset(df_train_ts_pred)ae_test = create_ae_dataset(df_test_ts_pred)print(f"AE Train: {len(ae_train)} | Test: {len(ae_test)}")print(f"Train labels: {ae_train['label'].value_counts().to_dict()}")print(f"Test labels: {ae_test['label'].value_counts().to_dict()}")

In [ ]:
import tensorflow as tffrom tensorflow.keras.models import Sequentialfrom tensorflow.keras.layers import LSTM, Dense, Dropout, Maskingfrom tensorflow.keras.preprocessing.sequence import pad_sequencesfrom sklearn.utils.class_weight import compute_class_weightMAX_TIMESTEPS = 50def prep_rnn(df):    s1 = pad_sequences(df['s1_prob_list'].values,maxlen=MAX_TIMESTEPS,padding='post',dtype='float32')    s2 = pad_sequences(df['s2_prob_list'].values,maxlen=MAX_TIMESTEPS,padding='post',dtype='float32')    return np.stack((s1,s2),axis=-1), df['label'].valuesXr_tr,yr_tr = prep_rnn(ae_train)Xr_te,yr_te = prep_rnn(ae_test)print(f"RNN shapes: train={Xr_tr.shape}, test={Xr_te.shape}")cl = np.unique(yr_tr)cw = {0:1.0,1:1.0}if len(cl)>1:    w = compute_class_weight('balanced',classes=cl,y=yr_tr)    cw = {cl[i]:w[i] for i in range(len(cl))}model = Sequential([    Masking(mask_value=0.,input_shape=(MAX_TIMESTEPS,2)),    LSTM(16), Dropout(0.3), Dense(1,activation='sigmoid')])model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),              loss='binary_crossentropy',metrics=['accuracy'])history = model.fit(Xr_tr,yr_tr,epochs=20,batch_size=4,                    class_weight=cw,validation_data=(Xr_te,yr_te),verbose=1)fig,(a1,a2) = plt.subplots(1,2,figsize=(14,5))ep = range(1,len(history.history['loss'])+1)a1.plot(ep,history.history['loss'],'b-o',ms=4,label='Train')a1.plot(ep,history.history['val_loss'],'r-o',ms=4,label='Val')a1.set_title('LSTM Loss'); a1.legend(); a1.grid(True,alpha=0.3)a2.plot(ep,history.history['accuracy'],'b-o',ms=4,label='Train')a2.plot(ep,history.history['val_accuracy'],'r-o',ms=4,label='Val')a2.set_title('LSTM Accuracy'); a2.legend(); a2.grid(True,alpha=0.3)plt.tight_layout(); plt.show()

---## 14. Conclusion### Key improvement v3.1: Type-Aware IP Split- v2/v3 split per-stage → seluruh `password` & `mitm` bisa jatuh di satu set- v3.1 split per-type → setiap attack type dijamin ada di train DAN test- Whale filtering tetap berlaku (IP besar wajib train)### Expected results:- Stage 1: improved via undersampling + per-flow features + threshold- Stage 2: preserved/improved karena semua type ada di train- Stage 3: improved via undersampling + threshold tuning